In [1]:
# ============================================================
#  NOTEBOOK 10 — Subject-Calibrated ICG Correction
#  Stage 1: Does a per-subject constant offset fix the bias?
# ============================================================
import numpy as np, pandas as pd
from pathlib import Path

FINAL = Path('./output/final_dataset')

# ---- Load ALL subjects (train+val+test).
# No cross-subject fitting happens in Stage 1, so there is no leakage
# concern here and no reason to exclude the locked test subjects.
rows = []
for split in ['train', 'val', 'test']:
    d = FINAL / split
    b   = np.load(d/'biases.npy')          # [b_bias, x_bias, pep_bias, lvet_bias]
    yp  = np.load(d/'y_pep.npy')
    yl  = np.load(d/'y_lvet.npy')
    m   = pd.read_csv(d/'meta.csv')
    df  = m.copy()
    df['split']    = split
    df['gt_pep']   = yp
    df['gt_lvet']  = yl
    df['icg_pep']  = yp + b[:, 2]          # bias = icg - gt  =>  icg = gt + bias
    df['icg_lvet'] = yl + b[:, 3]
    rows.append(df)

df = pd.concat(rows, ignore_index=True)
# Chronological order within subject (proxy: record, then beat index)
df = df.sort_values(['subject_id', 'record_id', 'beat_idx']).reset_index(drop=True)
print(f'{len(df)} beats, {df.subject_id.nunique()} subjects\n')


def metrics(pred, true):
    e = np.abs(pred - true)
    return dict(mae=e.mean(), w7=100*np.mean(e < 7), w15=100*np.mean(e < 15), n=len(e))


# ---- Theoretical floor: within-subject SD of the bias ----
# A constant-offset correction cannot do better than this subject's own
# beat-to-beat scatter. For roughly normal errors, MAE ~= 0.8 * SD.
print('=== WITHIN-SUBJECT BIAS SCATTER (the floor for this method) ===')
print(f'{"subj":<8}{"n":>5}{"PEP bias mean":>15}{"PEP SD":>9}{"->MAE":>8}'
      f'{"LVET bias mean":>16}{"LVET SD":>10}{"->MAE":>8}')
for s, g in df.groupby('subject_id'):
    dp = g.icg_pep - g.gt_pep
    dl = g.icg_lvet - g.gt_lvet
    print(f'{s:<8}{len(g):>5}{dp.mean():>15.1f}{dp.std():>9.1f}{0.8*dp.std():>8.1f}'
          f'{dl.mean():>16.1f}{dl.std():>10.1f}{0.8*dl.std():>8.1f}')


# ---- Calibration sweep ----
def calibrate(df, target, k, min_eval=3):
    """First k beats estimate the offset; evaluate on the rest."""
    icg_c, gt_c = f'icg_{target}', f'gt_{target}'
    preds, trues, per_subj, skipped = [], [], [], []
    for s, g in df.groupby('subject_id'):
        if len(g) < k + min_eval:
            skipped.append(s); continue
        cal, ev = g.iloc[:k], g.iloc[k:]
        b_hat = (cal[icg_c] - cal[gt_c]).mean()        # subject's offset
        p = (ev[icg_c] - b_hat).values
        t = ev[gt_c].values
        preds.append(p); trues.append(t)
        per_subj.append({'subject': s, 'n_eval': len(ev), 'b_hat': b_hat, **metrics(p, t)})
    return np.concatenate(preds), np.concatenate(trues), pd.DataFrame(per_subj), skipped


for target in ['pep', 'lvet']:
    icg_c, gt_c = f'icg_{target}', f'gt_{target}'
    print(f'\n{"="*66}\n  {target.upper()}\n{"="*66}')

    # Reference points
    raw    = metrics(df[icg_c].values, df[gt_c].values)
    glob   = metrics(df[icg_c].values - (df[icg_c] - df[gt_c]).mean(), df[gt_c].values)
    dummy  = metrics(np.full(len(df), df[gt_c].mean()), df[gt_c].values)
    # Oracle: offset from ALL of the subject's own beats (upper bound, not achievable)
    orc = df.groupby('subject_id').apply(
        lambda g: pd.Series({'p': g[icg_c] - (g[icg_c]-g[gt_c]).mean(), 't': g[gt_c]}))
    orc_p = np.concatenate([v for v in orc['p']]); orc_t = np.concatenate([v for v in orc['t']])
    oracle = metrics(orc_p, orc_t)

    print(f'{"method":<34}{"MAE":>8}{"±7ms":>9}{"±15ms":>9}{"n":>7}')
    print('-'*67)
    for name, m in [('Raw ICG (no correction)', raw),
                    ('Global offset (subject-blind)', glob),
                    ('Population mean (dummy)', dummy)]:
        print(f'{name:<34}{m["mae"]:>8.2f}{m["w7"]:>8.1f}%{m["w15"]:>8.1f}%{m["n"]:>7}')
    print('-'*67)
    for k in [1, 2, 3, 5, 10]:
        p, t, ps, sk = calibrate(df, target, k)
        m = metrics(p, t)
        note = f'  (skipped {len(sk)}: {",".join(sk)})' if sk else ''
        print(f'{"Calibrated, k=" + str(k):<34}{m["mae"]:>8.2f}{m["w7"]:>8.1f}%'
              f'{m["w15"]:>8.1f}%{m["n"]:>7}{note}')
        if k == 3:
            per_subj_k3 = ps
    print('-'*67)
    print(f'{"Oracle offset (all beats, ceiling)":<34}{oracle["mae"]:>8.2f}'
          f'{oracle["w7"]:>8.1f}%{oracle["w15"]:>8.1f}%{oracle["n"]:>7}')

    print(f'\n  Per-subject breakdown at k=3:')
    print(per_subj_k3.to_string(index=False,
          float_format=lambda x: f'{x:.1f}'))

279 beats, 13 subjects

=== WITHIN-SUBJECT BIAS SCATTER (the floor for this method) ===
subj        n  PEP bias mean   PEP SD   ->MAE  LVET bias mean   LVET SD   ->MAE
CHC01       9            7.2     12.2     9.8            25.4      14.4    11.5
CHC02      18          134.3     33.3    26.7          -125.0      51.5    41.2
CHC04      13          110.3     58.6    46.9           -76.1      91.9    73.6
CHC05      18           15.8     22.5    18.0           -81.2      25.4    20.3
CHC06       6            4.9      5.7     4.5             2.5       9.0     7.2
CHC07      22           36.1     52.4    41.9             9.4      53.5    42.8
CHC08       8           84.8     73.9    59.1           -64.6      75.5    60.4
CHC09      17           80.1     58.3    46.6           -32.6      68.3    54.6
CHC10      33           11.7     38.4    30.7            32.4      39.6    31.7
CHC12      22           -1.8     24.3    19.5            47.3      33.0    26.4
CHC14      44           16.7    

C:\Users\acer\AppData\Local\Temp\ipykernel_16192\1116031892.py:78: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  orc = df.groupby('subject_id').apply(
C:\Users\acer\AppData\Local\Temp\ipykernel_16192\1116031892.py:78: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  orc = df.groupby('subject_id').apply(


In [2]:
import numpy as np, pandas as pd
preds = np.load('./output/results/rq5_cross_modal_attention/loso_preds.npy')
true  = np.load('./output/results/rq5_cross_modal_attention/loso_true.npy')
meta  = pd.concat([pd.read_csv(f'./output/final_dataset/{s}/meta.csv') for s in ['train','val']],
                  ignore_index=True)
subj = meta['subject_id'].values
for i, name in enumerate(['PEP','LVET']):
    rs = []
    for s in np.unique(subj):
        m = (subj == s) & (true[:,i] != 0)
        if m.sum() >= 5 and np.std(true[m,i]) > 1e-6 and np.std(preds[m,i]) > 1e-6:
            rs.append(np.corrcoef(preds[m,i], true[m,i])[0,1])
    print(f'{name}: mean within-subject r = {np.mean(rs):+.3f} (n={len(rs)} subjects)')

PEP: mean within-subject r = +0.043 (n=10 subjects)
LVET: mean within-subject r = +0.105 (n=10 subjects)


In [3]:
# Same within-subject tracking check, run on RQ3's CNN-LSTM (ICG-only) predictions.
# If this also comes out near zero, the claim generalizes: no method tested
# (simple CNN-LSTM or the cross-modal attention model) tracks within-subject
# beat-to-beat variation — not just the attention model specifically.
import numpy as np, pandas as pd
preds_rq3 = np.load('./output/results/rq3_single_modality/cnnlstm_loso_preds.npy')
true_rq3  = np.load('./output/results/rq3_single_modality/cnnlstm_loso_true.npy')
meta_rq3  = pd.concat([pd.read_csv(f'./output/final_dataset/{s}/meta.csv') for s in ['train','val']],
                      ignore_index=True)
subj_rq3 = meta_rq3['subject_id'].values
for i, name in enumerate(['PEP','LVET']):
    rs = []
    for s in np.unique(subj_rq3):
        m = (subj_rq3 == s) & (true_rq3[:,i] != 0)
        if m.sum() >= 5 and np.std(true_rq3[m,i]) > 1e-6 and np.std(preds_rq3[m,i]) > 1e-6:
            rs.append(np.corrcoef(preds_rq3[m,i], true_rq3[m,i])[0,1])
    print(f'{name}: mean within-subject r = {np.mean(rs):+.3f} (n={len(rs)} subjects)')


PEP: mean within-subject r = -0.111 (n=10 subjects)
LVET: mean within-subject r = +0.213 (n=10 subjects)
